Converts HDF5 files to numpy matrices.

Run this notebook before anything else. 

## Specify folders and magnetization

In [ ]:
import numpy as np
# For folder names and simulation file structure refer to </README.md>

folder_name = r"F:\Contents\2024-08-20 Array noise" # The folder where all CST simulation files are located
m_mark = np.array([1,+1j,0])  # rotation of magnetization; we choose [1,+j,0]

## Run

In [ ]:
from rx_array_snr_evaluation import field
from importlib import reload
import itertools
import skrf

reload(field)

<module 'rx_array_snr_evaluation.field' from 'd:\\Wang_Wenjun\\Python\\array-noise-match-computation\\.venv\\Lib\\site-packages\\rx_array_snr_evaluation\\field.py'>

In [3]:
# There are nine simulation files
# Generates all folder names containing simulation results
# 12ch, outer_radius=134.00, loop_xy=97.60
# 12ch, outer_radius=171.28, loop_xy=120.57
# 12ch, outer_radius=208.56, loop_xy=143.93
#16ch, outer_radius=134.00, loop_xy=73.73
# 16ch, outer_radius=171.28, loop_xy=91.43
# 16ch, outer_radius=208.56, loop_xy=109.02
# 8ch, outer_radius=134.00, loop_xy=146.11
# 8ch, outer_radius=171.28, loop_xy=181.24
# 8ch, outer_radius=208.56, loop_xy=213.48

ch_n = np.array([8, 12, 16], dtype=np.int_)
sizes = np.array([134.00, 171.28, 208.56], dtype=np.float_)
loop_xys = np.array([[146.11, 181.24, 213.48], 
                     [97.60, 120.57, 143.93], 
                     [73.73, 91.43, 109.02]])
subfolder_names = [[], [], []]

# Generates simulation HD5 file results. Refer to </README.md>
for iter_m, iter_n in itertools.product([0, 1, 2], [0, 1, 2]):
    subfolder_name = f"{folder_name:s}\\{ch_n[iter_m]:d}ch, outer_radius={sizes[iter_n]:.2f}, loop_xy={loop_xys[iter_m, iter_n]:.2f}"
    subfolder_names[iter_m].append(subfolder_name)
hdf5_names = []
for ch in ch_n:
    hdf5_name = []
    for iter_n in range(ch):
        hdf5_name.append(f"H_Antenna {iter_n+1:d}.h5")
    hdf5_names.append(hdf5_name)
c = 1

# Transistors: Noiseless, Elcry2-u (elcry.com), Infineon BFP740, Infineon BFR35AP
# Transistor noise parameters: F_{opt}—minimum noise figure (linear scale), Z_{opt}—optimal noise impedance, R_{no}—noise resistance, just a parameter
# Some transistor parameters are not listed in datasheets; they are extracted from ADS simulation. Just accept them as they are. 
# BFP740 3 V, 6 mA; BFR35AP 10 V, 10 mA
transistor_names = ["Noiseless", "Elcry2-u", "BFP740", "BFR35AP"]
f_opt = 10 ** (np.array([0, 0.406208, 0.43869, 1.11818]) / 10)  
z_opt = np.array([50, 57.306557 + 21.569136j, 123.67115 + 3.12944j, 62.49675 + 2.55972j])
r_no = np.array([0, 1.926394, 6.15520, 8.01097], dtype=np.float_)
g_noise = (r_no / np.abs(z_opt) ** 2).astype(np.float_)  # Rn / z_opt ** 2, equivalent to matrix [D]
lange_N = np.real(z_opt) * g_noise

def mask_ROI(position: np.ndarray, _h: np.ndarray) -> np.ndarray: 
    """ Defines region of interest
        Now it's a circle centered at (0, 0), radius 100.
    """
    return np.linalg.norm(position[:, 0:2], axis=1) <= 100.

In [4]:
subfolder_names

[['F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11',
  'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=171.28, loop_xy=181.24',
  'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=208.56, loop_xy=213.48'],
 ['F:\\Contents\\2024-08-20 Array noise\\12ch, outer_radius=134.00, loop_xy=97.60',
  'F:\\Contents\\2024-08-20 Array noise\\12ch, outer_radius=171.28, loop_xy=120.57',
  'F:\\Contents\\2024-08-20 Array noise\\12ch, outer_radius=208.56, loop_xy=143.93'],
 ['F:\\Contents\\2024-08-20 Array noise\\16ch, outer_radius=134.00, loop_xy=73.73',
  'F:\\Contents\\2024-08-20 Array noise\\16ch, outer_radius=171.28, loop_xy=91.43',
  'F:\\Contents\\2024-08-20 Array noise\\16ch, outer_radius=208.56, loop_xy=109.02']]

In [5]:
hdf5_names

[['H_Antenna 1.h5',
  'H_Antenna 2.h5',
  'H_Antenna 3.h5',
  'H_Antenna 4.h5',
  'H_Antenna 5.h5',
  'H_Antenna 6.h5',
  'H_Antenna 7.h5',
  'H_Antenna 8.h5'],
 ['H_Antenna 1.h5',
  'H_Antenna 2.h5',
  'H_Antenna 3.h5',
  'H_Antenna 4.h5',
  'H_Antenna 5.h5',
  'H_Antenna 6.h5',
  'H_Antenna 7.h5',
  'H_Antenna 8.h5',
  'H_Antenna 9.h5',
  'H_Antenna 10.h5',
  'H_Antenna 11.h5',
  'H_Antenna 12.h5'],
 ['H_Antenna 1.h5',
  'H_Antenna 2.h5',
  'H_Antenna 3.h5',
  'H_Antenna 4.h5',
  'H_Antenna 5.h5',
  'H_Antenna 6.h5',
  'H_Antenna 7.h5',
  'H_Antenna 8.h5',
  'H_Antenna 9.h5',
  'H_Antenna 10.h5',
  'H_Antenna 11.h5',
  'H_Antenna 12.h5',
  'H_Antenna 13.h5',
  'H_Antenna 14.h5',
  'H_Antenna 15.h5',
  'H_Antenna 16.h5']]

In [9]:
# Generates signal vector results, and save them into corresponding folders

signal_generators = [[], [], []]
for iter_m, iter_n in itertools.product([0, 1, 2], [0, 1, 2]):
    file_list = []
    for iter_o in range(len(hdf5_names[iter_m])):
        file_list.append(f"{subfolder_names[iter_m][iter_n]:s}\\{hdf5_names[iter_m][iter_o]:s}")
    print(file_list)
    signal_generator = field.SignalVectorGenerator.from_files(file_list, mask_ROI)
    print(signal_generator._position.shape)
    signal_generator.save(subfolder_names[iter_m][iter_n] + "\\signal_generator_data")
    signal_generators[iter_m].append(signal_generator)

['F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 1.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 2.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 3.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 4.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 5.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 6.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 7.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=134.00, loop_xy=146.11\\H_Antenna 8.h5']
(785343, 3)
['F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=171.28, loop_xy=181.24\\H_Antenna 1.h5', 'F:\\Contents\\2024-08-20 Array noise\\8ch, outer_radius=171.28, loop_xy=181.24\\H_Antenna 2.h5', 'F:\\C

In [10]:
array_pos_roi0 = signal_generator._position
signal_generator._position.shape

(785343, 3)

In [13]:
# Generates impedance matrices. This is not the impedance matrix of the antenna array, but the core part of (39) of Findeklee's "Array Noise Matching", doi:10.1109/TAP.2010.2096183.
# "Core part" means the big bracket between U_s^H and U_s. You can control whether inverse sign (^{-1}) will be included in or excluded from the core; by default it is included. 

# Generates many intermediate matrices and classes
z_generators = []
noise_match_cores = []
signal_opt_cores = []
admissibilities = []
snr_mappers = []
for iter_m, iter_n, iter_o in itertools.product([0, 1, 2], [0, 1, 2], range(len(f_opt))):
    snp_net = skrf.Network(subfolder_names[iter_m][iter_n] + f"\\matZ_{ch_n[iter_m]}_size{iter_n+1}.s{ch_n[iter_m]}p")
    array_Z = snp_net.z[0, ...]
    z_generator = field.ImpedanceMatrixGenerator(array_Z, c, f_opt[iter_o], lange_N[iter_o])
    del snp_net
    noise_match_core = z_generator.noise_match_core()
    us = signal_generators[iter_m][iter_n].signal_vectors(m_mark)  # 
    signal_opt_core, us_admissibility = z_generator.signal_optimization_core(us)
    # Then shows admissible proportion
    print(f"{ch_n[iter_m]} channels, size {iter_n+1}, {transistor_names[iter_o]}: Admissible proportion = {np.sum(us_admissibility) / np.size(us_admissibility):%}")

    # signal_center_core = signal_opt_core[center_location_filtered, ...]#[np.newaxis, ...]
    snr_mapper = field.SNRMapper(us, array_pos_roi0, noise_match_core, 
                                second_ref_core_39=z_generator._inv_A)
    z_generators.append(z_generator)
    admissibilities.append(us_admissibility)
    signal_opt_cores.append(signal_opt_core)
    snr_mappers.append(snr_mapper)
    noise_match_cores.append(noise_match_core)

8 channels, size 1, Noiseless: Admissible proportion = 10.010912%
8 channels, size 1, Elcry2-u: Admissible proportion = 10.010912%
8 channels, size 1, BFP740: Admissible proportion = 10.010912%
8 channels, size 1, BFR35AP: Admissible proportion = 10.010912%
8 channels, size 2, Noiseless: Admissible proportion = 5.108214%
8 channels, size 2, Elcry2-u: Admissible proportion = 5.108214%
8 channels, size 2, BFP740: Admissible proportion = 5.108214%
8 channels, size 2, BFR35AP: Admissible proportion = 5.108214%
8 channels, size 3, Noiseless: Admissible proportion = 1.114163%
8 channels, size 3, Elcry2-u: Admissible proportion = 1.114163%
8 channels, size 3, BFP740: Admissible proportion = 1.114163%
8 channels, size 3, BFR35AP: Admissible proportion = 1.114163%
12 channels, size 1, Noiseless: Admissible proportion = 2.033506%
12 channels, size 1, Elcry2-u: Admissible proportion = 2.033506%
12 channels, size 1, BFP740: Admissible proportion = 2.033506%
12 channels, size 1, BFR35AP: Admissible